In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Configurazione: un solo output finale, i file intermedi vengono eliminati.
INPUT_FILE = Path("../cambiamenti_climatici/matteo/dataset_pulito3.csv")
TMP_LAG = Path("_tmp_lag_features.csv")
TMP_TARGET = Path("_tmp_target_future.csv")
FINAL_OUTPUT = Path("dataset_pulito3_lag_target_final.csv")
LEGACY_OUTPUTS = [
    Path("dataset_pulito3_lag_features.csv"),
    Path("dataset_pulito3_target_future.csv"),
]

for path in (TMP_LAG, TMP_TARGET, FINAL_OUTPUT, *LEGACY_OUTPUTS):
    path.unlink(missing_ok=True)

FINAL_OUTPUT


In [ ]:
# 1) Crea le lag features temporanee.
!{sys.executable} create_lag_features.py --input "{INPUT_FILE}" --output "{TMP_LAG}"

# 2) Crea i target futuri temporanei.
!{sys.executable} create_future_targets.py --input "{INPUT_FILE}" --output "{TMP_TARGET}"

# 3) Unisce tutto su cella geografica + giorno e salva un solo CSV finale.
lag_df = pd.read_csv(TMP_LAG)
target_df = pd.read_csv(TMP_TARGET)

target_cols = [
    "date", "lat_cell", "lon_cell",
    "fire_next_1d", "fire_next_3d", "fire_next_7d", "fire_count_next_7d",
]

final_df = lag_df.merge(
    target_df[target_cols],
    on=["date", "lat_cell", "lon_cell"],
    how="inner",
    validate="one_to_one",
)

if len(final_df) != len(lag_df) or len(final_df) != len(target_df):
    raise ValueError("Merge non allineato: controlla chiavi o righe duplicate.")

final_df.to_csv(FINAL_OUTPUT, index=False)

# 4) Rimuove i file intermedi: in cartella resta solo il CSV finale.
TMP_LAG.unlink(missing_ok=True)
TMP_TARGET.unlink(missing_ok=True)

print(f"Creato: {FINAL_OUTPUT} | shape={final_df.shape}")
final_df.head()
